In [1]:
import numpy as np
import pandas as pd

from astroML.datasets import fetch_sdss_specgals
from sklearn.model_selection import train_test_split
from astroML.utils.decorators import pickle_results

In [2]:
data = data = fetch_sdss_specgals()
df = pd.DataFrame(data)

pd.set_option('display.float_format', lambda x: '%.9f' % x)

In [3]:
# 1. Calculate the Color Indices
df['u-g'] = df['modelMag_u'] - df['modelMag_g']
df['g-r'] = df['modelMag_g'] - df['modelMag_r']
df['r-i'] = df['modelMag_r'] - df['modelMag_i']
df['i-z'] = df['modelMag_i'] - df['modelMag_z']

# 2. Clean the Data
model_mag_error = 5
df_clean = df[
    # Ensure the magnitude errors are low
    (df['modelMagErr_u'] < model_mag_error) &
    (df['modelMagErr_g'] < model_mag_error) &
    (df['modelMagErr_r'] < model_mag_error) &
    (df['modelMagErr_i'] < model_mag_error) &
    (df['modelMagErr_z'] < model_mag_error) &
    
    # Ignore the local universe blob to see the higher redshift track better
    (df['z'] > 0.1)
]

df_clean = df
df_clean.shape

(661598, 47)

dataset v1:
- model_mag_error = 1
- z > 0.25

Dataset v2:
- model_mag_error = 2.5
- z > 0.2

Dataset v3:
- model_mag_error = 5
- z > 0.1

Dataset v4: 
df_clean = df (no data cleaning)

In [4]:
# train/val/test split

X = df_clean[['u-g', 'g-r', 'r-i', 'i-z']]
y = df_clean['z']

from astroML.utils.decorators import pickle_results
from sklearn.model_selection import train_test_split

# Use a permanent, fresh filename that doesn't have old cached data attached to it
@pickle_results('data/dataset_split_v1.pkl')
def split_data(X_data, y_data):
    X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

@pickle_results: computing results and saving to 'data/dataset_split_v4.pkl'
